# 📋 Rapport Candidat — Démo du module `candidate_report.py`

Ce notebook montre comment le pipeline RAG Neurosymbolique génère un **feedback structuré**
pour un candidat après sa réponse à un cas ECG.

**Le rapport comprend :**
1. 🔍 **Analyse du texte** — concepts extraits par l'IA (NER + Juge)
2. 📊 **Note & explication** — score dégressif par génération, détail par validant
3. 📝 **Éléments descriptifs** — concepts attendus non notés
4. 🟢 **Découvertes** — concepts vrais ajoutés par le candidat (hors barème)

| Cellule | Description |
|---------|-------------|
| 1 | Setup — imports & initialisation moteur RAG |
| 2 | Chargement du Golden Set (15 cas) |
| 3 | 🧪 Cas 3 : Fibrillation atriale — Réponse parfaite |
| 4 | 🧪 Cas 3 : FA — Réponse partielle (child match) |
| 5 | 🧪 Cas 7 : BBD complet — Réponse riche avec découvertes |
| 6 | 🧪 Cas 9 : BAV 2 Mobitz 2 — Le cas difficile |
| 7 | 🧪 Mode interactif — Testez votre propre texte |

In [1]:
# ============================================================
# CELLULE 1 — Setup & Initialisation
# ============================================================
import sys, os, json, importlib
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path(r"C:\Users\Administrateur\bmad\ECG lecture")
EVAL_ROOT    = Path(r"C:\Users\Administrateur\bmad\ECG evaluation")
RAG_ROOT     = Path(r"C:\Users\Administrateur\bmad\RAG ontologique")

sys.path.insert(0, str(RAG_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")
os.chdir(str(RAG_ROOT))

# Import & reload du module de rapport
import candidate_report; importlib.reload(candidate_report)
from candidate_report import (
    generate_candidate_report, format_report_text, format_report_html,
    CandidateReport,
)
from scoring import SCORE_BY_GENERATION, SCORE_GENERATION_FLOOR

print(f"[OK] OPENAI_API_KEY : {'✅' if os.getenv('OPENAI_API_KEY') else '❌'}")
print(f"[OK] Barème dégressif : Gen1={SCORE_BY_GENERATION[1]:.0f}%, Gen2={SCORE_BY_GENERATION[2]:.0f}%, Gen3={SCORE_BY_GENERATION[3]:.0f}%, Gen4+={SCORE_GENERATION_FLOOR:.0f}%")
print(f"🚀 Module candidate_report prêt.")

[OK] OPENAI_API_KEY : ✅
[OK] Barème dégressif : Gen1=90%, Gen2=80%, Gen3=70%, Gen4+=60%
🚀 Module candidate_report prêt.


In [2]:
# ============================================================
# CELLULE 2 — Chargement du Golden Set
# ============================================================
from scoring import find_owl_concept

golden_cases = {}
for case_dir in sorted((EVAL_ROOT / "goldenset").iterdir()):
    meta_file = case_dir / "metadata.json"
    if case_dir.is_dir() and meta_file.exists():
        with open(meta_file, 'r', encoding='utf-8') as f:
            meta = json.load(f)
        case_num = int(meta['name'])
        
        # Construire golden_names, golden_ids, golden_roles
        names, ids, roles = [], [], []
        for ann in meta.get('annotations', []):
            owl = find_owl_concept(ann['concept'])
            if owl:
                names.append(ann['concept'])
                ids.append(owl['ontology_id'])
                role = 'validant' if 'validant' in ann.get('annotation_role', '').lower() else 'descripteur'
                roles.append(role)
        
        golden_cases[case_num] = {
            'diagnostic_principal': meta.get('diagnostic_principal', ''),
            'golden_names': names,
            'golden_ids': ids,
            'golden_roles': roles,
        }

print(f"[GOLD] {len(golden_cases)} cas chargés :")
for num, case in sorted(golden_cases.items()):
    validants = [n for n, r in zip(case['golden_names'], case['golden_roles']) if r == 'validant']
    descripteurs = [n for n, r in zip(case['golden_names'], case['golden_roles']) if r == 'descripteur']
    print(f"   Cas {num:2d} │ {case['diagnostic_principal']:<50s} │ V={len(validants)} D={len(descripteurs)} │ {validants}")

[GOLD] 15 cas chargés :
   Cas  1 │ ECG normal                                         │ V=1 D=1 │ ['ECG normal']
   Cas  2 │ BAV complet                                        │ V=1 D=1 │ ['BAV complet']
   Cas  3 │ Fibrillation atriale                               │ V=1 D=1 │ ['Fibrillation atriale']
   Cas  4 │ Microvoltage                                       │ V=1 D=3 │ ['Microvoltage']
   Cas  5 │ Hyperkaliémie                                      │ V=2 D=0 │ ['Hyperkaliémie', 'BAV de haut grade']
   Cas  6 │ Stimulation atriale                                │ V=2 D=1 │ ['Stimulation atriale', 'Bloc fasciculaire antérieur gauche']
   Cas  7 │ Bloc de branche droit complet                      │ V=3 D=0 │ ['Bloc de branche droit complet', 'BAV de type 1', 'Bloc fasciculaire antérieur gauche']
   Cas  8 │ Flutter droit typique                              │ V=1 D=0 │ ['Flutter droit typique']
   Cas  9 │ BAV 2 Mobitz 2                                     │ V=1 D=2 │ ['BAV 2 Mobi

## 🧪 Cas 3 : Fibrillation atriale — Réponse parfaite

Le candidat identifie exactement le bon diagnostic. Score attendu : **100%**.

In [3]:
# ============================================================
# CELLULE 3 — Cas 3 : FA — Réponse parfaite
# ============================================================
from IPython.display import display, HTML

cas = golden_cases[3]

report = generate_candidate_report(
    texte_etudiant="fibrillation atriale",
    golden_names=cas['golden_names'],
    golden_ids=cas['golden_ids'],
    golden_roles=cas['golden_roles'],
    diagnostic_principal=cas['diagnostic_principal'],
)

# Affichage texte (pour le terminal)
print(format_report_text(report))

════════════════════════════════════════════════════════════════════════════════
📋 RAPPORT D'ÉVALUATION — Fibrillation atriale
════════════════════════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────────────────────────
🔍 SECTION 1 — Analyse de votre texte (1 concepts identifiés)
────────────────────────────────────────────────────────────────────────────────
   Votre texte : « fibrillation atriale »

   1. [✓] « fibrillation atriale »  →  Fibrillation atriale  (coupe_circuit)

────────────────────────────────────────────────────────────────────────────────
📊 SECTION 2 — Votre note : 100.0% (1/1 diagnostics validants)
────────────────────────────────────────────────────────────────────────────────
   Barème : Exact=100% | Gen1=90% | Gen2=80% | Gen3=70% | Gen4+=60% | Hypothèse=×0.8

   ✅ Vous avez identifié « Fibrillation atriale » — correspondance exacte.

   ══ NOTE FINALE : 100.0% ══

──────────────────────────────

In [4]:
# Affichage HTML (dark theme)
display(HTML(format_report_html(report)))

## 🧪 Cas 3 : FA — Réponse partielle (child match)

Le candidat mentionne des signes de FA (trémulation, irrégulier) sans nommer le diagnostic.
Le scoring hiérarchique devrait donner un **CHILD gen1 = 90%**.

In [5]:
# ============================================================
# CELLULE 4 — Cas 3 : FA — Réponse partielle
# ============================================================
cas = golden_cases[3]

report_partial = generate_candidate_report(
    texte_etudiant="trémulation de la ligne de base, rythme irrégulier, QRS fins, tachycardie",
    golden_names=cas['golden_names'],
    golden_ids=cas['golden_ids'],
    golden_roles=cas['golden_roles'],
    diagnostic_principal=cas['diagnostic_principal'],
)

display(HTML(format_report_html(report_partial)))

## 🧪 Cas 7 : Bloc de branche droit complet — Réponse riche

3 validants attendus (BBD complet + BAV1 + HBAG). Le candidat donne une réponse
détaillée avec des découvertes additionnelles.

In [6]:
# ============================================================
# CELLULE 5 — Cas 7 : BBD complet — Réponse riche
# ============================================================
cas7 = golden_cases[7]

report_bbd = generate_candidate_report(
    texte_etudiant="rythme sinusal, bloc de branche droit complet, hemibloc anterieur gauche, bav 1, qrs larges, axe gauche",
    golden_names=cas7['golden_names'],
    golden_ids=cas7['golden_ids'],
    golden_roles=cas7['golden_roles'],
    diagnostic_principal=cas7['diagnostic_principal'],
)

display(HTML(format_report_html(report_bbd)))

## 🧪 Cas 9 : BAV 2 Mobitz 2 — Le cas difficile

C'est le cas le plus dur du benchmark (60% de moyenne). Le candidat confond Mobitz 1 et 2.

In [7]:
# ============================================================
# CELLULE 6 — Cas 9 : BAV 2 Mobitz 2 — Erreur classique
# ============================================================
cas9 = golden_cases[9]

# Scénario A : Le candidat confond Mobitz 1 et 2
print("\n🔴 SCÉNARIO A : Confusion Mobitz 1 / Mobitz 2")
print("─" * 60)
report_mobitz_err = generate_candidate_report(
    texte_etudiant="bav 2 mobitz 1",
    golden_names=cas9['golden_names'],
    golden_ids=cas9['golden_ids'],
    golden_roles=cas9['golden_roles'],
    diagnostic_principal=cas9['diagnostic_principal'],
)
display(HTML(format_report_html(report_mobitz_err)))


🔴 SCÉNARIO A : Confusion Mobitz 1 / Mobitz 2
────────────────────────────────────────────────────────────


In [8]:
# Scénario B : Le candidat donne la bonne réponse complète
print("\n🟢 SCÉNARIO B : Réponse correcte et complète")
print("─" * 60)
report_mobitz_ok = generate_candidate_report(
    texte_etudiant="BAV 2 Mobitz 2, bloc de branche droit, HBPG",
    golden_names=cas9['golden_names'],
    golden_ids=cas9['golden_ids'],
    golden_roles=cas9['golden_roles'],
    diagnostic_principal=cas9['diagnostic_principal'],
)
display(HTML(format_report_html(report_mobitz_ok)))


🟢 SCÉNARIO B : Réponse correcte et complète
────────────────────────────────────────────────────────────


## 🧪 Mode interactif — Testez votre propre texte

Choisissez un cas et tapez votre réponse pour voir le rapport.

In [9]:
# ============================================================
# CELLULE 7 — Mode interactif
# ============================================================

# ── Paramètres à modifier ─────────────────────────────────────
NUMERO_CAS = 8          # 👈 Changez le numéro de cas (1-15)
TEXTE_CANDIDAT = """    
flutter commun 2/1 qrs fins
"""                      # 👈 Tapez votre réponse ici
# ──────────────────────────────────────────────────────────────

cas_interactif = golden_cases[NUMERO_CAS]
print(f"📌 Cas {NUMERO_CAS} — {cas_interactif['diagnostic_principal']}")
print(f"   Golden : {cas_interactif['golden_names']}")
print(f"   Rôles  : {cas_interactif['golden_roles']}")
print()

report_interactif = generate_candidate_report(
    texte_etudiant=TEXTE_CANDIDAT.strip(),
    golden_names=cas_interactif['golden_names'],
    golden_ids=cas_interactif['golden_ids'],
    golden_roles=cas_interactif['golden_roles'],
    diagnostic_principal=cas_interactif['diagnostic_principal'],
)

display(HTML(format_report_html(report_interactif)))

📌 Cas 8 — Flutter droit typique
   Golden : ['Flutter droit typique']
   Rôles  : ['validant']



In [10]:
# ============================================================
# CELLULE 8 — Version texte du rapport interactif (optionnel)
# ============================================================
print(format_report_text(report_interactif))

════════════════════════════════════════════════════════════════════════════════
📋 RAPPORT D'ÉVALUATION — Flutter droit typique
════════════════════════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────────────────────────
🔍 SECTION 1 — Analyse de votre texte (3 concepts identifiés)
────────────────────────────────────────────────────────────────────────────────
   Votre texte : « flutter commun 2/1 qrs fins »

   1. [✓] « flutter commun »  →  Flutter droit typique  (coupe_circuit)
   2. [✓] « 2/1 »  →  2/1  (coupe_circuit)
   3. [✓] « QRS fins »  →  QRS fins  (coupe_circuit)

────────────────────────────────────────────────────────────────────────────────
📊 SECTION 2 — Votre note : 100.0% (1/1 diagnostics validants)
────────────────────────────────────────────────────────────────────────────────
   Barème : Exact=100% | Gen1=90% | Gen2=80% | Gen3=70% | Gen4+=60% | Hypothèse=×0.8

   ✅ Vous avez identifié « Flutter dro

## 🎓 Section 6 — Test du Feedback Pédagogique (Cours SFC, Item 231)

Le feedback pédagogique est généré automatiquement par GPT-4o à partir des extraits du cours SFC.
- **Pour chaque erreur** : rappel du cours, piège classique, rang EDN
- **Format** : Intégré directement dans le rapport HTML (Section 5)
- **Paramètre** : `with_feedback=True` (défaut) ou `False` pour les benchmarks

## 📊 Panorama complet — 10 cas supplémentaires

Test systématique sur les cas **1, 2, 4, 5, 6, 10, 11, 12, 13, 14** avec des textes étudiants
de niveaux variés (parfait, partiel, erreur). Le cas 15 est couvert par le cas 10 (même diag BBG).

| Cas | Diagnostic | Texte étudiant simulé | Niveau attendu |
|-----|-----------|----------------------|----------------|
| 1 | ECG normal | Parfait | 100% |
| 2 | BAV complet | Confusion BAV complet / BAV haut degré | Partiel |
| 4 | Microvoltage | Réponse correcte | 100% |
| 5 | Hyperkaliémie | Oubli du BAV de haut grade | Partiel |
| 6 | Stimulation atriale | Bonne mais imprécise (pacemaker) | Partiel? |
| 10 | BBG complet | Réponse correcte | 100% |
| 11 | TV | Réponse correcte | 100% |
| 12 | SCA ST+ | Réponse correcte | 100% |
| 13 | WPW | Confusion avec BBG | 0% |
| 14 | ESV | Réponse correcte | 100% |

In [11]:
# ============================================================
# PANORAMA — 10 cas supplémentaires (exécution en boucle)
# ============================================================
from IPython.display import display, HTML
import importlib, time
import candidate_report; importlib.reload(candidate_report)
from candidate_report import generate_candidate_report, format_report_text, format_report_html

# ── Définition des 10 cas test ──────────────────────────────────
PANORAMA = {
    1: {
        "texte": "ECG normal, rythme sinusal, FC 75 bpm, axe normal, pas de trouble de repolarisation",
        "commentaire": "✅ Réponse parfaite et détaillée",
    },
    2: {
        "texte": "BAV de haut degré, dissociation auriculo-ventriculaire, bradycardie",
        "commentaire": "🟠 Confusion BAV complet ↔ BAV haut degré (parent gen1?)",
    },
    4: {
        "texte": "microvoltage diffus",
        "commentaire": "✅ Réponse correcte, concise",
    },
    5: {
        "texte": "hyperkaliémie, ondes T amples pointues symétriques",
        "commentaire": "🟠 Hyperkaliémie trouvée mais oubli du BAV de haut grade",
    },
    6: {
        "texte": "pacemaker auriculaire, hémibloc antérieur gauche, QRS fins",
        "commentaire": "🟠 Pacemaker ≈ stimulation ? HBAG attendu comme validant",
    },
    10: {
        "texte": "bloc de branche gauche complet, axe gauche, QRS larges",
        "commentaire": "✅ Réponse parfaite",
    },
    11: {
        "texte": "tachycardie ventriculaire monomorphe, QRS larges, FC 180",
        "commentaire": "✅ TV identifiée correctement",
    },
    12: {
        "texte": "SCA ST+ antérieur étendu, sus-décalage V1-V6 D1 aVL",
        "commentaire": "✅ SCA ST+ identifié",
    },
    13: {
        "texte": "bloc de branche gauche incomplet, PR court",
        "commentaire": "❌ Confusion WPW ↔ BBG incomplet — erreur classique",
    },
    14: {
        "texte": "extrasystoles ventriculaires monomorphes, rythme sinusal sous-jacent",
        "commentaire": "✅ ESV identifiées correctement",
    },
}

# ── Exécution en boucle ─────────────────────────────────────────
results = {}
t_global = time.time()

for num, test in sorted(PANORAMA.items()):
    cas = golden_cases[num]
    t0 = time.time()
    
    report = generate_candidate_report(
        texte_etudiant=test["texte"],
        golden_names=cas['golden_names'],
        golden_ids=cas['golden_ids'],
        golden_roles=cas['golden_roles'],
        diagnostic_principal=cas['diagnostic_principal'],
        with_feedback=False,  # sans feedback GPT pour aller vite
    )
    
    dt = time.time() - t0
    results[num] = {"report": report, "dt": dt, "test": test}
    
    # Résumé rapide
    v_found = report.nb_validants_trouves
    v_total = report.nb_validants_attendus
    score = report.score_final_pct
    emoji = "🟢" if score >= 90 else "🟠" if score >= 50 else "🔴"
    
    print(f"{emoji} Cas {num:2d} │ {cas['diagnostic_principal']:<55s} │ {score:5.1f}% │ V={v_found}/{v_total} │ {dt:.1f}s │ {test['commentaire']}")

print(f"\n⏱️ Total : {time.time()-t_global:.1f}s pour 10 cas")

🟢 Cas  1 │ ECG normal                                              │ 100.0% │ V=1/1 │ 3.9s │ ✅ Réponse parfaite et détaillée


🟢 Cas  2 │ BAV complet                                             │ 100.0% │ V=1/1 │ 4.5s │ 🟠 Confusion BAV complet ↔ BAV haut degré (parent gen1?)
🟢 Cas  4 │ Microvoltage                                            │ 100.0% │ V=1/1 │ 0.9s │ ✅ Réponse correcte, concise
🟠 Cas  5 │ Hyperkaliémie                                           │  50.0% │ V=1/2 │ 3.1s │ 🟠 Hyperkaliémie trouvée mais oubli du BAV de haut grade


⚠️  Le Juge a renvoyé un ID invalide : 'WANDERING_PACEMAKER'. IDs valides : {'BLOC_AURICULO_VENTRICULAIRE', '"WANDERING_PACEMAKER"', 'BLOC_INTERATRIAL', 'ARYTHMIE_ATRIALE'}. Forçage → NONE.


🟠 Cas  6 │ Stimulation atriale                                     │  50.0% │ V=1/2 │ 3.9s │ 🟠 Pacemaker ≈ stimulation ? HBAG attendu comme validant
🟢 Cas 10 │ Bloc de branche gauche complet                          │ 100.0% │ V=1/1 │ 3.3s │ ✅ Réponse parfaite
🟢 Cas 11 │ Tachycardie ventriculaire                               │ 100.0% │ V=1/1 │ 5.9s │ ✅ TV identifiée correctement
🟢 Cas 12 │ Syndrome coronarien à la phase aigue avec sus-décalage du segment ST │ 100.0% │ V=1/1 │ 13.9s │ ✅ SCA ST+ identifié
🔴 Cas 13 │ Faisceau accessoire à conduction antérograde            │   0.0% │ V=0/1 │ 1.2s │ ❌ Confusion WPW ↔ BBG incomplet — erreur classique
🟢 Cas 14 │ Extrasystole ventriculaire                              │ 100.0% │ V=1/1 │ 5.5s │ ✅ ESV identifiées correctement

⏱️ Total : 46.1s pour 10 cas


In [12]:
# ============================================================
# PANORAMA — Affichage détaillé HTML de chaque cas
# ============================================================
# Cliquez sur chaque cas pour voir le rapport complet

for num in sorted(results.keys()):
    r = results[num]
    report = r["report"]
    test = r["test"]
    cas = golden_cases[num]
    score = report.score_final_pct
    emoji = "🟢" if score >= 90 else "🟠" if score >= 50 else "🔴"
    
    # Construire détail validants
    detail_lines = []
    for vd in report.validant_details:
        icon = "✅" if vd.found and vd.match_type == "exact" else "🟠" if vd.found else "❌"
        detail_lines.append(f"{icon} {vd.golden_name} → {vd.match_type} ({vd.score_pct:.0f}%)")
    validant_detail = "<br>".join(detail_lines) if detail_lines else "—"
    
    # Concepts extraits
    concepts = ", ".join([f"{c.terme_brut}→{c.concept_name}" for c in report.concepts_extraits if c.ontology_id != "NONE"])
    if not concepts:
        concepts = "<em>aucun résolu</em>"
    
    # Découvertes
    decouvertes = ", ".join([d.concept_name for d in report.decouvertes]) if report.decouvertes else "—"
    
    html = f"""
    <details style="background:#1e1e1e; border-radius:8px; padding:12px; margin:8px 0; border-left:4px solid {'#4CAF50' if score>=90 else '#FF9800' if score>=50 else '#F44336'};">
        <summary style="cursor:pointer; color:#e0e0e0; font-size:15px;">
            <strong>{emoji} Cas {num} — {cas['diagnostic_principal']}</strong>
            &nbsp;&nbsp;│&nbsp;&nbsp;
            <span style="color:{'#4CAF50' if score>=90 else '#FF9800' if score>=50 else '#F44336'}; font-weight:bold;">{score:.0f}%</span>
            &nbsp;&nbsp;│&nbsp;&nbsp;
            <span style="color:#999; font-size:13px;">V={report.nb_validants_trouves}/{report.nb_validants_attendus}</span>
            &nbsp;&nbsp;│&nbsp;&nbsp;
            <span style="color:#888; font-size:12px;">{test['commentaire']}</span>
        </summary>
        <div style="padding:12px; color:#ccc; font-size:13px; line-height:1.8;">
            <div style="margin-bottom:8px;">
                <strong style="color:#90CAF9;">📝 Texte étudiant :</strong><br>
                <em style="color:#bbb;">« {test['texte']} »</em>
            </div>
            <div style="margin-bottom:8px;">
                <strong style="color:#FFD54F;">🎯 Validants attendus :</strong><br>
                {validant_detail}
            </div>
            <div style="margin-bottom:8px;">
                <strong style="color:#81C784;">🔗 Concepts résolus :</strong><br>
                {concepts}
            </div>
            <div>
                <strong style="color:#00BCD4;">🟢 Découvertes :</strong> {decouvertes}
            </div>
        </div>
    </details>
    """
    display(HTML(html))

In [13]:
# ============================================================
# CELLULE 9 — Reload modules & test feedback pédagogique
# ============================================================
import importlib
import edn_knowledge_base; importlib.reload(edn_knowledge_base)
import pedagogical_feedback; importlib.reload(pedagogical_feedback)
import candidate_report; importlib.reload(candidate_report)
from candidate_report import generate_candidate_report, format_report_text, format_report_html
from IPython.display import display, HTML

# Cas 3 : FA — réponse partielle (le candidat oublie la repolarisation précoce)
cas3 = golden_cases[3]
print(f"📌 Cas 3 — {cas3['diagnostic_principal']}")
print(f"   Golden : {cas3['golden_names']}")
print(f"   Rôles  : {cas3['golden_roles']}")
print()

# Le candidat ne mentionne que la FA, pas la repolarisation précoce → feedback attendu
report_feedback = generate_candidate_report(
    texte_etudiant="fibrillation atriale qrs fins tachycardie",
    golden_names=cas3['golden_names'],
    golden_ids=cas3['golden_ids'],
    golden_roles=cas3['golden_roles'],
    diagnostic_principal=cas3['diagnostic_principal'],
    with_feedback=True,  # ← active le feedback pédagogique
)

# Afficher le feedback brut
if report_feedback.feedback_pedagogique:
    fb = report_feedback.feedback_pedagogique
    print(f"✅ Feedback généré ({len(fb.texte)} chars)")
    print(f"   Rangs manqués : {fb.rang_edn_manques}")
    print(f"   Concepts cités : {fb.concepts_cours_cites}")
    print(f"   Concept rang A manqué ? {'⚠️ OUI' if fb.has_critical_miss else 'Non'}")
    print(f"\n{'─'*60}")
    print(fb.texte)
else:
    print("⚠️ Pas de feedback généré")

📌 Cas 3 — Fibrillation atriale
   Golden : ['Fibrillation atriale', 'Repolarisation précoce']
   Rôles  : ['validant', 'descripteur']

✅ Feedback généré (2273 chars)
   Rangs manqués : []
   Concepts cités : ['Fibrillation atriale']
   Concept rang A manqué ? Non

────────────────────────────────────────────────────────────
## 1. Référence au cours

Pour le concept de **fibrillation atriale**, il est classé A (indispensable) : 📖 « La fibrillation atriale correspond à une activation atriale anarchique. C'est une tachycardie entre 100 et 200 bpm à QRS irrégulièrement irréguliers. » — (Item 231, SFC). Les points clés à retenir incluent le fait que la FA est le seul diagnostic en cas de tachycardie complètement irrégulière à QRS fins, et que l'activation atriale est anarchique, entraînant des intervalles RR non multiples d'une valeur commune. Maîtriser ce concept est crucial pour les EDN, car il s'agit d'un diagnostic fréquent en pratique clinique.

## 2. Votre interprétation

Bravo pour v

In [14]:
# ============================================================
# CELLULE 10 — Rapport HTML complet avec feedback pédagogique
# ============================================================
# Le feedback pédagogique apparaît en Section 5 du rapport HTML

display(HTML(format_report_html(report_feedback)))

In [15]:
# ============================================================
# CELLULE 11 — Test feedback sur cas BAV (erreur classique)
# ============================================================
# Le candidat confond BAV 2 Mobitz 2 avec BAV 1 → piège classique

cas9 = golden_cases[9]
print(f"📌 Cas 9 — {cas9['diagnostic_principal']}")
print(f"   Golden : {cas9['golden_names']}")
print()

report_bav_err = generate_candidate_report(
    texte_etudiant="bav du premier degré rythme sinusal bradycardie",
    golden_names=cas9['golden_names'],
    golden_ids=cas9['golden_ids'],
    golden_roles=cas9['golden_roles'],
    diagnostic_principal=cas9['diagnostic_principal'],
    with_feedback=True,
)

# Le feedback devrait mentionner le piège BAV1 vs BAV2
display(HTML(format_report_html(report_bav_err)))

📌 Cas 9 — BAV 2 Mobitz 2
   Golden : ['BAV 2 Mobitz 2', 'Bloc de branche droit', 'Bloc fasciculaire postérieur gauche']



In [ ]:
# ============================================================
# CELLULE 12 — Test with_feedback=False (mode benchmark)
# ============================================================
# Vérifie que le rapport fonctionne sans feedback (plus rapide)

import time
t0 = time.time()

report_no_fb = generate_candidate_report(
    texte_etudiant="fibrillation atriale qrs fins",
    golden_names=cas3['golden_names'],
    golden_ids=cas3['golden_ids'],
    golden_roles=cas3['golden_roles'],
    diagnostic_principal=cas3['diagnostic_principal'],
    with_feedback=False,  # ← PAS de feedback pédagogique
)

t_no_fb = time.time() - t0

print(f"✅ Rapport sans feedback : {t_no_fb:.1f}s")
print(f"   Score : {report_no_fb.score_final_pct:.1f}%")
print(f"   Feedback : {report_no_fb.feedback_pedagogique}")  # Should be None
print(f"\n   → Temps économisé : pas d'appel GPT supplémentaire pour le feedback")